# Integration Test — Russell's RAG Output → Youssef's Verdict Engine

Separate notebook dedicated to running and inspecting the full end-to-end pipeline test.
Loads `integration_test_cases.json` (Russell's 13 test claims) and runs each through `VerdictEngine`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../system2b"))

In [2]:
import os
print(os.getcwd())

/Users/russelltamer/Desktop/system 2 RAG/system2a_rag


In [3]:
import json
import pandas as pd
from agents.verdict_engine import VerdictEngine

pd.set_option('display.max_colwidth', 60)

with open("integration_test_cases.json", encoding="utf-8") as f:
    cases = json.load(f)

print(f"Loaded {len(cases)} test cases")

Loaded 13 test cases


In [4]:
print("Initialising Verdict Engine (loads NLI model, ~2.8GB first run)...")
engine = VerdictEngine()
print("Ready.")

Initialising Verdict Engine (loads NLI model, ~2.8GB first run)...
[NLI] Loading model... (one-time, ~2.8GB)
[NLI] Using device: cpu


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[NLI] Model ready.
Ready.


In [5]:
rows = []
raw_results = []

for case in cases:
    claim = case.get("claim", "<unknown>")
    signal = case.get("verdict_signal", "<unknown>")
    n_a = len(case.get("bucket_a", []))
    n_b = len(case.get("bucket_b", []))
    try:
        result = engine.decide(case)
        rows.append({
            "claim": claim,
            "signal": signal,
            "bucket_a_hits": n_a,
            "bucket_b_hits": n_b,
            "final_verdict": result["final_verdict"],
            "confidence": round(result["confidence"], 2),
            "n_stances": len(result.get("stance_breakdown", [])),
            "reason": result["reason"][:80],
        })
        raw_results.append(result)
    except Exception as e:
        rows.append({
            "claim": claim, "signal": signal, "bucket_a_hits": n_a, "bucket_b_hits": n_b,
            "final_verdict": "ERROR", "confidence": None, "n_stances": None, "reason": str(e),
        })
        raw_results.append(None)

df = pd.DataFrame(rows)
df

,claim,signal,bucket_a_hits,bucket_b_hits,final_verdict,confidence,n_stances,reason
0,السيسي باع سيناء للإسرائيليين,HIGH_FAKE_MATCH,3,0,FALSE,0.89,0,Russell found a high-similarity match (0.885) to a known...
1,لقاح أسترازينيكا يسبب جلطات دموية,HIGH_TRUE_MATCH,3,0,TRUE,0.87,0,Russell found a high-similarity match (0.872) to a known...
2,نظام السيسي قدم لمجلس النواب تشريع يشرع بيع قناة السويس,HIGH_FAKE_MATCH,3,0,FALSE,0.91,0,Russell found a high-similarity match (0.908) to a known...
3,مفيش دليل على ان فيروس كورونا اتعمل في مختبر,HIGH_FAKE_MATCH,3,0,FALSE,0.87,0,Russell found a high-similarity match (0.871) to a known...
4,مش صحيح ان اللقاح بيخلي الناس تموت,HIGH_FAKE_MATCH,3,0,FALSE,0.86,0,Russell found a high-similarity match (0.860) to a known...
5,الحكومة بتكدب على الناس في موضوع الكهرباء,HIGH_TRUE_MATCH,3,0,TRUE,0.86,0,Russell found a high-similarity match (0.860) to a known...
6,إسرائيل تصعد عملياتها العسكرية في قطاع غزة,POSSIBLE_MATCH,3,5,TRUE,0.95,4,Russell found a possible match; evidence is ambiguous. A...
7,أسعار النفط ارتفعت بسبب التوترات في منطقة الخليج,EVIDENCE_FOUND,3,5,TRUE,0.95,3,Russell found relevant evidence in the knowledge base. A...
8,الجامعة العربية عقدت اجتماعاً طارئاً لبحث الأزمة السورية,LOW_CONFIDENCE,3,5,UNVERIFIED,0.70,0,[LOW_CONFIDENCE + Solid Bucket A] Russell reported low r...
9,مصر وقعت اتفاقية تعاون اقتصادي مع الصين,EVIDENCE_FOUND,3,5,TRUE,0.95,1,Russell found relevant evidence in the knowledge base. A...


## Flag Cases Worth Inspecting

Specifically: any `LOW_CONFIDENCE` input that produced a *high* output confidence (contradiction),
and any `POSSIBLE_*` signal where `bucket_b_hits == 0` (the empty-evidence gap we found).

In [6]:
suspicious = df[
    ((df["signal"] == "LOW_CONFIDENCE") & (df["confidence"] > 0.7)) |
    ((df["signal"].str.startswith("POSSIBLE")) & (df["bucket_b_hits"] == 0))
]
suspicious

,claim,signal,bucket_a_hits,bucket_b_hits,final_verdict,confidence,n_stances,reason


In [7]:
print(raw_results[8]['reason'])
print('---')
print(raw_results[10]['reason'])

[LOW_CONFIDENCE + Solid Bucket A] Russell reported low retrieval confidence, but a high-similarity fact-check match was found (similarity 0.826, source: Misbar). The fact-checked verdict (UNVERIFIED) is retained, but confidence is capped at 0.7 due to low retrieval reliability. ⚠️ Russell reported low retrieval confidence, but solid fact-checked evidence was found.
---
[LOW_CONFIDENCE + Solid Bucket A] Russell reported low retrieval confidence, but a high-similarity fact-check match was found (similarity 0.839, source: Fatabayyano). The fact-checked verdict (UNVERIFIED) is retained, but confidence is capped at 0.7 due to low retrieval reliability. ⚠️ Russell reported low retrieval confidence, but solid fact-checked evidence was found.


In [8]:
# Drill into a specific case by row index — change idx to inspect any case in detail
idx = 8  # change this to inspect a different row
print("INPUT:")
print(json.dumps(cases[idx], ensure_ascii=False, indent=2))
print("\nOUTPUT:")
print(json.dumps(raw_results[idx], ensure_ascii=False, indent=2))

INPUT:
{
  "claim": "الجامعة العربية عقدت اجتماعاً طارئاً لبحث الأزمة السورية",
  "dialect": "BEI",
  "dialect_confidence": 0.598,
  "query_used": "الجامعة العربية عقدت اجتماعاً طارئاً لبحث الأزمة السورية",
  "verdict_signal": "LOW_CONFIDENCE",
  "final_verdict": "UNVERIFIED",
  "confidence": 0.2,
  "bucket_a": [
    {
      "claim": "عاجل: الجزائر تعتذر عن استضافة القمة العربية إن لم تحضرها سورية.",
      "similarity": 0.826,
      "label": "PARTLY_FALSE",
      "source": "Misbar",
      "dialect": "",
      "debunk": "تتداول صفحات وحسابات على موقع التواصل الاجتماعي فيسبوك، حديثًا، خبرًا مفاده أن الجزائر تعتذر عن استضافة القمة العربية التي من المنتظر عقدها في البلاد، إن لم تحضرها سورية."
    },
    {
      "claim": "بايدن من منبر الأمم المتحدة منذ قليل : مستعدون لرفع كامل العقوبات عن سوريا",
      "similarity": 0.822,
      "label": "FALSE",
      "source": "AFP",
      "dialect": "",
      "debunk": "زعم مستخدمون لمواقع التواصل الاجتماعي أنّ الرئيس الأميركي جو بايدن أعلن خلال كلمته ا

In [10]:
from agents.verdict_engine import _normalise_rerank, _RERANK_MIN, _RERANK_MAX

print(f"System 3's expected input range: [0, 1]")
print(f"System 2's actual rerank_score range: [{_RERANK_MIN}, {_RERANK_MAX}]")
print()

real_scores = [-1.4758, -3.1463, -4.5697, -5.0837, -5.7557]
print(f"{'Raw rerank_score':>20} | {'Normalized (System 3 input)':>28}")
for s in real_scores:
    print(f"{s:>20.4f} | {_normalise_rerank(s):>28.4f}")

System 3's expected input range: [0, 1]
System 2's actual rerank_score range: [-8.0, 10.0]

    Raw rerank_score |  Normalized (System 3 input)
             -1.4758 |                       0.3625
             -3.1463 |                       0.2697
             -4.5697 |                       0.1906
             -5.0837 |                       0.1620
             -5.7557 |                       0.1247
